# Marketplace Funnel & Activation Analysis

This notebook contains the reproducible analytical trail behind the portfolio summary. It separates observed patterns from hypotheses and uses user-level denominators unless stated otherwise.

In [1]:
from pathlib import Path
import json, sys
import duckdb
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency
import statsmodels.formula.api as smf
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.metrics import user_funnel, funnel_summary
RAW = ROOT / 'data/raw'
users = pd.read_csv(RAW/'users.csv', parse_dates=['signup_date'])
events = pd.read_csv(RAW/'events.csv', parse_dates=['event_timestamp'])
orders = pd.read_csv(RAW/'orders.csv', parse_dates=['order_timestamp'])
len(users), len(events), len(orders)

(40000, 468253, 25472)

## 1. Metric contract

- **User purchase rate:** unique purchasers / acquired users. It measures acquisition quality without rewarding repeat orders.
- **Exploration rate:** users with a listing or offer view / acquired users.
- **Checkout completion:** purchasing users / users who started checkout. This isolates the post-checkout step.
- **60-day repeat rate:** users with a second purchase within 60 days / first-time buyers with a full 60-day observation window.

The funnel is non-strict at user level: a user is counted if they reached a stage at least once. Session-level checkout analysis is used separately when diagnosing friction.

In [2]:
flags = user_funnel(events)
funnel = funnel_summary(flags)
funnel.style.format({'share_of_acquired_users':'{:.1%}', 'step_conversion':'{:.1%}'})

,stage,users,share_of_acquired_users,step_conversion
0,session_start,39885,100.0%,100.0%
1,listing_view,37686,94.5%,94.5%
2,offer_view,29148,73.1%,77.3%
3,checkout_start,21024,52.7%,72.1%
4,purchase,16257,40.8%,77.3%


## 2. Acquisition quality

Traffic volume and downstream value are evaluated separately. A growing channel can increase the numerator of visits while reducing the portfolio's average conversion rate through mix shift.

In [3]:
profile = users.merge(flags, on='user_id', how='left')
profile[['session_start','listing_view','offer_view','checkout_start','purchase']] = profile[['session_start','listing_view','offer_view','checkout_start','purchase']].fillna(0).astype(int)
value = orders.groupby('user_id').agg(orders=('order_id','nunique'), revenue=('order_value','sum')).reset_index()
profile = profile.merge(value, on='user_id', how='left').fillna({'orders':0,'revenue':0})
channel = profile.groupby('acquisition_channel').agg(acquired_users=('user_id','size'), purchasers=('purchase','sum'), revenue=('revenue','sum')).assign(purchase_rate=lambda x:x.purchasers/x.acquired_users, revenue_per_user=lambda x:x.revenue/x.acquired_users).sort_values('purchase_rate', ascending=False)
channel.style.format({'purchase_rate':'{:.1%}', 'revenue_per_user':'£{:.2f}', 'revenue':'£{:,.0f}'})

,acquired_users,purchasers,revenue,purchase_rate,revenue_per_user
acquisition_channel,,,,,
direct,8432,4349,"£658,038",51.6%,£78.04
organic_search,9627,4460,"£628,193",46.3%,£65.25
referral,4852,2111,"£281,593",43.5%,£58.04
paid_search,7486,3161,"£428,352",42.2%,£57.22
paid_social,9603,2176,"£220,568",22.7%,£22.97


In [4]:
profile['signup_month'] = profile.signup_date.dt.to_period('M').astype(str)
cohorts = profile.groupby(['signup_month','acquisition_channel']).agg(users=('user_id','size'), purchase_rate=('purchase','mean')).reset_index()
cohorts.pivot(index='signup_month', columns='acquisition_channel', values='users')

acquisition_channel,direct,organic_search,paid_search,paid_social,referral
signup_month,,,,,
2024-01,920,1075,796,390,598
2024-02,1308,1465,1177,728,758
2024-03,1613,1619,1337,1291,843
2024-04,1551,1810,1346,1768,898
2024-05,1565,1819,1468,2353,939
2024-06,1475,1839,1362,3073,816


**Interpretation.** The aggregate cohort decline should not be read as every channel deteriorating. The channel mix changes. This is a composition effect and motivates channel-standardized monitoring.

## 3. Device friction after checkout

Comparing purchase rates by device mixes acquisition intent with interface effects. Checkout completion provides a closer behavioral comparison, although it remains observational.

In [5]:
sessions = events.assign(checkout=events.event_name.eq('checkout_start'), purchase_event=events.event_name.eq('purchase')).groupby(['session_id','device_type']).agg(checkout=('checkout','max'), purchase=('purchase_event','max')).reset_index()
checkout = sessions[sessions.checkout].groupby('device_type').agg(checkouts=('checkout','sum'), purchases=('purchase','sum')).assign(completion=lambda x:x.purchases/x.checkouts)
checkout.style.format({'completion':'{:.1%}'})

,checkouts,purchases,completion
device_type,,,
desktop,13170,11122,84.4%
mobile,21665,14350,66.2%


**What the data shows:** mobile checkout starters finish less often. **Hypotheses:** form friction, payment-method coverage, slower performance, or a different context of use. Device assignment is not randomized, so the gap supports prioritizing instrumentation and an experiment—not a causal claim.

## 4. Early behavior and activation

Offer comparison is measured in the first session, before the subsequent purchase outcome. A multivariable logistic model checks whether the association survives observed channel, device, country, and cohort differences.

In [6]:
first_sessions = events.groupby(['user_id','session_id']).event_timestamp.min().reset_index().sort_values('event_timestamp').drop_duplicates('user_id')[['user_id','session_id']]
early = events.merge(first_sessions).groupby('user_id').event_name.apply(lambda x:(x=='offer_view').sum()).rename('offer_views_first_session')
model_df = profile.merge(early, on='user_id', how='left')
model_df['compared_two_offers'] = model_df.offer_views_first_session.ge(2)
model_df['purchased'] = model_df.purchase.astype(int)
activation = model_df.groupby('compared_two_offers').agg(users=('user_id','size'), purchase_rate=('purchased','mean'), repeat_rate=('orders',lambda x:(x>=2).mean()))
activation.style.format({'purchase_rate':'{:.1%}','repeat_rate':'{:.1%}'})

,users,purchase_rate,repeat_rate
compared_two_offers,,,
False,32544,38.1%,13.3%
True,7456,51.8%,19.9%


In [7]:
model = smf.logit('purchased ~ compared_two_offers + C(acquisition_channel) + C(device_type) + C(country) + C(signup_month)', data=model_df).fit(disp=False)
term = 'compared_two_offers[T.True]'
pd.Series({'adjusted_odds_ratio':np.exp(model.params[term]), 'ci_low':np.exp(model.conf_int().loc[term,0]), 'ci_high':np.exp(model.conf_int().loc[term,1]), 'p_value':model.pvalues[term]})

adjusted_odds_ratio    1.723833e+00
ci_low                 1.635723e+00
ci_high                1.816688e+00
p_value                5.349048e-92
dtype: float64

**Boundary on inference.** The adjusted association is consistent with offer comparison identifying high-intent users. It does not establish that forcing extra offer views will cause conversion. Unobserved intent remains a plausible common cause.

## 5. Category and retention checks

In [8]:
category_sessions = events.groupby(['session_id','category']).event_name.agg(list).reset_index()
category_sessions['explored'] = category_sessions.event_name.apply(lambda x: 'listing_view' in x or 'offer_view' in x)
category_sessions['purchased'] = category_sessions.event_name.apply(lambda x: 'purchase' in x)
category_sessions.groupby('category').agg(exploring_sessions=('explored','sum'), purchases=('purchased','sum')).assign(purchase_per_exploration=lambda x:x.purchases/x.exploring_sessions).sort_values('purchase_per_exploration')

,exploring_sessions,purchases,purchase_per_exploration
category,,,
fashion,19137,4442,0.232116
collectibles,16266,4397,0.270318
home,17115,5901,0.344785
services,12915,4539,0.351452
electronics,17096,6193,0.362248


In [9]:
ranked = orders.sort_values('order_timestamp').copy()
ranked['order_number'] = ranked.groupby('user_id').cumcount()+1
ranked['next_order'] = ranked.groupby('user_id').order_timestamp.shift(-1)
first = ranked[(ranked.order_number==1) & (ranked.order_timestamp <= '2024-05-01')].copy()
first['repeat_60d'] = first.next_order.le(first.order_timestamp + pd.Timedelta(days=60))
first.repeat_60d.mean()

0.3544947904046523

## 6. Decision

1. Diagnose and test the mobile checkout gap first because it occurs closest to value and has a direct interface hypothesis.
2. Add channel-quality metrics (30-day purchase rate and revenue per acquired user) to acquisition reviews; volume alone is misleading.
3. Treat early offer comparison as an activation signal. Test a decision-support intervention rather than assuming the observed behavior is causal.

The proposed experiment in the README targets mobile checkout simplification, with completed purchase per checkout starter as the primary metric and payment failure, refund, order value, and desktop spillover as guardrails.